# 02 - Exploratory Data Analysis (EDA) & Credit Risk Drivers

This notebook investigates key drivers of credit default (`SeriousDlqin2yrs`), bivariate risk profiles, delinquency history, and the statistical impact of unverified or missing applicant income.

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is accessible
sys.path.append('..')
from src.data_loader import load_raw_data

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')

df = load_raw_data()
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns.")

## 1. Credit Utilization vs Default Rate

Revolving credit card utilization is a primary risk factor in consumer lending. Utilization exceeding 80% or 100% signals liquidity stress.

In [ ]:
df['Util_Bucket'] = pd.cut(
    df['RevolvingUtilizationOfUnsecuredLines'],
    bins=[-np.inf, 0.3, 0.7, 1.0, 2.0, np.inf],
    labels=['< 30%', '30-70%', '70-100%', '100-200%', '> 200%']
)
util_risk = df.groupby('Util_Bucket', observed=False)['SeriousDlqin2yrs'].agg(['count', 'mean'])
util_risk['default_rate_pct'] = util_risk['mean'] * 100
display(util_risk)

## 2. Delinquency History Analysis

Analysis of 30-59, 60-89, and 90+ days past due counts, along with anomaly code verification (codes 96 and 98).

In [ ]:
delinq_cols = ['NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfTimes90DaysLate']
for col in delinq_cols:
    print(f"Anomalous values (>= 96) in {col}:", (df[col] >= 96).sum())

# Clean delinquency and examine default rates
clean_late90 = df['NumberOfTimes90DaysLate'].replace([96, 98], np.nan)
display(df.groupby(clean_late90)['SeriousDlqin2yrs'].agg(['count', 'mean']))

## 3. Impact of Missing Income

Evaluating default propensity when income is missing vs reported.

In [ ]:
df['Income_Missing'] = df['MonthlyIncome'].isnull()
income_comp = df.groupby('Income_Missing')['SeriousDlqin2yrs'].agg(['count', 'mean'])
income_comp['default_rate_pct'] = income_comp['mean'] * 100
display(income_comp)